# Chapter 10 — An all-sky search with Gaia

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/10_gaia_allsky.ipynb)

**Goal.** Gaia observes the whole sky, uniformly enough to search all of it at once for compact stellar overdensities.
Koposov, Belokurov & Torrealba (2017, MNRAS 470, 2702, https://ui.adsabs.harvard.edu/abs/2017MNRAS.470.2702K) did this
with Gaia DR1 — positions and G magnitudes only — and discovered two new star clusters, Gaia 1 and Gaia 2. We reproduce
their all-sky significance map (their Figure 1) with DR1, then repeat it with DR3, and with DR3 plus cuts that DR1 could
not make: removing nearby stars with measured parallaxes, and a colour cut.

**You will learn**
1. to count ~10⁹ stars on a HEALPix grid *inside the database*, downloading only the counts;
2. to smooth a HEALPix map with a Gaussian kernel, and to compute a local mean and variance in an annulus, using
   spherical harmonics;
3. why several kernel sizes are needed (Gaia 2 is 2′ across, Crater 2 is 30′);
4. to compare data sets by the known objects they recover and the spurious peaks they produce.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('healpy') is None:                  # Colab does not ship healpy
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'healpy'], check=True)
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp

DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release once, check its SHA-256, return a DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    assert sha == MANIFEST[name]['sha256'], f'{path} does not match the manifest: delete it and run again'
    return pd.read_parquet(path)

## 10.1 Counting a billion stars without downloading them

Gaia DR1 has 1.1 × 10⁹ sources, DR3 1.8 × 10⁹. We need only how many stars fall in each pixel of a sky grid. A Gaia
`source_id` encodes the source's position: `source_id // 2**35` is its index in the nested HEALPix grid of level 12
(nside = 4096). Dividing by 2³⁹ instead gives level 10 (nside = 1024, pixels of 3.4′). So the database can build the
histogram with a `GROUP BY` and return ~12 million counts instead of ~10⁹ rows:

In [ ]:
print(MANIFEST['gaia_dr1_counts']['queries'][0])
print()
print(MANIFEST['gaia_dr3_counts']['queries'][0])

The selection follows Koposov et al. (2017): $17 < G < 21$ (brighter stars are mostly nearby foreground), and
`astrometric_excess_noise` below $10^{0.15(G - 15) + 0.25}$ mas, which removes most galaxies (an extended source scatters
Gaia's one-dimensional position measurements). The DR3 file has three counts per pixel:

- `n_k17`: the same selection as in DR1;
- `n_far`: minus stars whose parallax places them significantly within 1 kpc, $\varpi - 3\sigma_\varpi > 1$ mas
  (stars without a parallax are kept);
- `n_far_cmd`: also $0.3 < G_{\rm BP} - G_{\rm RP} < 1.3$, the colours of old, metal-poor turn-off and giant stars
  (stars without a colour are dropped).

In [ ]:
NSIDE = 1024
NPIX = hp.nside2npix(NSIDE)


def to_map(df, col):
    """Full-sky RING-ordered map of a count column (the files are in NESTED order, which healpy's smoothing does not use)."""
    m = np.zeros(NPIX)
    m[df.hpx.to_numpy()] = df[col].to_numpy()
    return hp.reorder(m, n2r=True)


dr1 = load('gaia_dr1_counts')
dr3 = load('gaia_dr3_counts')
maps = {'DR1': to_map(dr1, 'n_k17'), 'DR3': to_map(dr3, 'n_k17'),
        'DR3, far': to_map(dr3, 'n_far'), 'DR3, far + colour': to_map(dr3, 'n_far_cmd')}
del dr1, dr3
for k, m in maps.items():
    print(f'{k:18s}: {m.sum():.3e} stars')

fig = plt.figure(figsize=(14, 4))
hp.mollview(np.log10(np.maximum(maps['DR1'], 1)), fig=fig, sub=(1, 2, 1), title='Gaia DR1, 17 < G < 21: log10 counts per pixel',
            cmap='Greys', min=0.5, max=3)
hp.mollview(np.log10(np.maximum(maps['DR3'], 1)), fig=fig, sub=(1, 2, 2), title='Gaia DR3, same selection', cmap='Greys', min=0.5, max=3)

**Figure 10.1.** Star counts on the sky (equatorial coordinates, Mollweide projection). The Galactic disc and bulge
dominate; the Magellanic Clouds are at lower right. In DR1, the stripes are Gaia's scanning pattern: after only 14
months, some areas had been observed many more times than others, so the completeness at G ≈ 20–21 varies across the
sky. DR3 (34 months) is far more uniform.

## 10.2 Kernel density and significance on the sphere

Following Koposov et al. (2017), the density is the histogram $H$ convolved with a normalised Gaussian kernel,
$D = K * H$, and the significance is

$$ S = \frac{D - \langle D\rangle_A}{\sqrt{{\rm Var}_A(D)}}, $$

where the mean and variance are taken over an annulus $A$ around each point. Using the local variance rather than a
Poisson estimate absorbs the extra fluctuations of the scanning pattern. The paper does not give the annulus radii; we
use 3σ–6σ of the kernel.

One numerical safeguard: ${\rm Var}_A(D) = \langle D^2\rangle_A - \langle D\rangle_A^2$ is a difference of two smoothed
maps and can cancel to ~0 (or below) through numerical ringing when the kernel is not much larger than a pixel. In a first
version this produced S ~ 10⁵ in patches of the sky. We therefore never let the variance fall below the Poisson variance
of $D$, $\langle D\rangle_A\,\Omega_{\rm pix}/(4\pi\sigma^2)$, and use kernels of at least 4′ (pixels are 3.4′).

On the sphere, convolution with a function of angular distance is a multiplication in spherical-harmonic space:
`healpy.smoothing` does it for a Gaussian, and for the annulus we compute its harmonic coefficients from its radial
profile (`beam2bl`), normalised so that it averages.

In [ ]:
LMAX = 3 * NSIDE - 1
_theta = np.linspace(0, np.radians(5), 5000)


def annulus_bl(r1_deg, r2_deg):
    """Harmonic coefficients of a normalised top-hat annulus r1 < theta < r2."""
    b = ((_theta > np.radians(r1_deg)) & (_theta < np.radians(r2_deg))).astype(float)
    bl = hp.beam2bl(b, _theta, LMAX)
    return bl / bl[0]


OMEGA_PIX = 4 * np.pi / NPIX


def significance(H, sigma_arcmin):
    """S map for Gaussian kernel sigma and annulus 3-6 sigma (RING maps); variance floored at the Poisson value."""
    sig = np.radians(sigma_arcmin / 60)
    D = hp.smoothing(H, sigma=sig, iter=1)
    bl = annulus_bl(3 * sigma_arcmin / 60, 6 * sigma_arcmin / 60)
    mean = hp.alm2map(hp.almxfl(hp.map2alm(D, lmax=LMAX, iter=1), bl), NSIDE)
    msq = hp.alm2map(hp.almxfl(hp.map2alm(D**2, lmax=LMAX, iter=1), bl), NSIDE)
    var_poisson = np.maximum(mean, 0) * OMEGA_PIX / (4 * np.pi * sig**2)
    return ((D - mean) / np.sqrt(np.maximum(msq - mean**2, var_poisson) + 1e-12)).astype(np.float32)


S12 = significance(maps['DR1'], 12)
hp.mollview(np.clip(S12, -3, 8), title='Gaia DR1: significance of stellar overdensities, 12′ Gaussian kernel', cmap='viridis',
            min=-3, max=8, unit='S')

**Figure 10.2.** Our reconstruction of Figure 1 of Koposov et al. (2017). Compact yellow spots are known satellites and
star clusters; the fine network of filaments and arcs follows the DR1 scanning pattern, as described by Koposov et al.
Dividing by the local scatter removes the large-scale density gradients (the disc does not stand out), but the disc and
the Magellanic Clouds are still full of real small-scale structure (open clusters, dust) and are usually excluded from
halo searches.

## 10.3 One kernel is not enough

A kernel matched to the object's size maximises its significance. We compute S with σ = 4′, 6′ and 12′ (our pixels are
3.4′; the paper used kernels from 1.5′ to 24′ on a finer grid) and look at four objects of different size: the two Gaia
discoveries, Boötes I and the very extended Crater 2.

In [ ]:
objs = pd.DataFrame([('Gaia 1', 101.47, -16.75, 6.5), ('Gaia 2', 28.12, 53.04, 1.9), ('Boötes I', 210.02, 14.51, 10.0),
                     ('Crater 2', 177.31, -18.41, 31.2)], columns=['name', 'ra', 'dec', 'rh_arcmin'])
SIGMAS = (4, 6, 12)


def peak_at(S, ra, dec, radius_arcmin):
    """Largest S within radius of (ra, dec)."""
    return S[hp.query_disc(NSIDE, hp.ang2vec(ra, dec, lonlat=True), np.radians(radius_arcmin / 60))].max()


Smaps = {s: (S12 if s == 12 else significance(maps['DR1'], s)) for s in SIGMAS}
tab = pd.DataFrame({f'σ = {s}′': [peak_at(Smaps[s], o.ra, o.dec, 5) for o in objs.itertuples()] for s in SIGMAS}, index=objs.name)
print('max S within 5′ of each object, Gaia DR1:')
print(tab.round(1).to_string())

fig = plt.figure(figsize=(13, 3.5))
for k, o in enumerate(objs.itertuples()):
    for j, s in enumerate(SIGMAS):
        hp.gnomview(Smaps[s], rot=(o.ra, o.dec), xsize=120, reso=1.0, fig=fig, sub=(len(SIGMAS), len(objs), j * len(objs) + k + 1),
                    min=-3, max=8, cmap='viridis', cbar=False, notext=True, title=f'{o.name}, σ = {s}′', margins=(0.005,) * 4)
assert tab.loc['Gaia 1', 'σ = 4′'] > 6 and tab.loc['Gaia 2', 'σ = 4′'] > 5 and tab.loc['Crater 2', 'σ = 12′'] > 5

**Figure 10.3.** Significance maps (2° × 2°) around the four objects for the three kernels. Gaia 1 ($r_h \approx 6.5′$,
10′ from Sirius) and Gaia 2 ($r_h \approx 2′$) stand out only with the smallest kernel — Gaia 2 at S ≈ 5.6, below the
paper's 6σ threshold, because our 3.4′ pixels do not allow kernels as small as its 1.5′ ones; Crater 2 ($r_h \approx 31′$)
appears only with the largest. A search must scan sizes, and then calibrate for the many trials (Chapter 5).

## 10.4 DR1 versus DR3, and the value of parallaxes and colours

We now run the three kernels on the four maps and ask two questions, away from the Galactic plane ($|b| > 20°$):

1. **Recovery**: how many known Milky Way satellites and star clusters (Local Volume Database v1.1.1, `host = mw`) have
   S > 6 within max(5′, $r_h$) of their position, in any kernel?
2. **Spurious peaks**: how many pixels have S > 6 farther than 1° from any known object, excluding the Magellanic Clouds
   (within 12° of the LMC and 6° of the SMC)? (A rough measure: real searches also mask bright galaxies and bright stars.)

In [ ]:
lv = load('lvdb_mw')
rot = hp.Rotator(coord=['C', 'G'])
pix_ra, pix_dec = hp.pix2ang(NSIDE, np.arange(NPIX), lonlat=True)
pix_b = rot(pix_ra, pix_dec, lonlat=True)[1]
lv_b = rot(lv.ra.to_numpy(), lv.dec.to_numpy(), lonlat=True)[1]
lv = lv[np.abs(lv_b) > 20].reset_index(drop=True)
near_known = np.zeros(NPIX, bool)
for o in lv.itertuples():
    near_known[hp.query_disc(NSIDE, hp.ang2vec(o.ra, o.dec, lonlat=True), np.radians(1.0))] = True
pix_vec = np.array(hp.pix2vec(NSIDE, np.arange(NPIX)))
clouds = (pix_vec.T @ hp.ang2vec(80.894, -69.756, lonlat=True) > np.cos(np.radians(12))) | \
         (pix_vec.T @ hp.ang2vec(13.187, -72.829, lonlat=True) > np.cos(np.radians(6)))
high_b = (np.abs(pix_b) > 20) & ~clouds

summary, recovered = [], {}
for name, H in maps.items():
    Smax = np.max([Smaps[s] if name == 'DR1' else significance(H, s) for s in SIGMAS], axis=0)
    rec = np.array([peak_at(Smax, o.ra, o.dec, max(5, o.rhalf if np.isfinite(o.rhalf) else 5)) > 6 for o in lv.itertuples()])
    recovered[name] = rec
    summary.append(dict(data=name, recovered=int(rec.sum()), of=len(lv),
                        spurious_pixels=int(np.sum((Smax > 6) & high_b & ~near_known))))
    if name == 'DR3, far + colour':
        S_best = Smax
summary = pd.DataFrame(summary)
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
mv = lv.M_V.to_numpy()
d = 10**(lv.distance_modulus.to_numpy() / 5 + 1) / 1e3
for name, mk in zip(maps, 'osD^'):
    ax.scatter(d[recovered[name]], mv[recovered[name]], marker=mk, s=40, facecolors='none', label=f'{name}: {recovered[name].sum()}')
ax.scatter(d, mv, s=4, c='k', label='all known, |b| > 20°')
ax.set(xscale='log', xlabel='distance [kpc]', ylabel='$M_V$', title='known objects recovered (S > 6)')
ax.invert_yaxis()
ax.legend(fontsize=8)
plt.tight_layout()
hp.mollview(np.clip(S_best, -3, 8), title='Gaia DR3, far + colour: max S over 3′, 6′, 12′ kernels', cmap='viridis', min=-3, max=8, unit='S')

**Figure 10.4.** Top: known Milky Way satellites and clusters at $|b| > 20°$ in the plane of distance and absolute
magnitude (black dots), with those recovered at S > 6 by each version of the data. Bottom: the DR3 significance map with
the parallax and colour cuts.

## Discussion

- **DR3 versus DR1.** With the same selection, DR3 recovers somewhat more known objects than DR1 (49 against 45 of 145
  at $|b| > 20°$): more stars per object, from deeper completeness at G ≈ 20–21 after 34 instead of 14 months. It also has
  *more* S > 6 pixels away from catalogued objects (~2000 against ~1200). Whether these are uncatalogued real systems,
  artefacts around bright stars and galaxies, or residual scanning structure, we have not established here — the vetting
  of Chapters 3 and 9 would be the next step (exercise).
- **Parallax cut.** Removing stars significantly closer than 1 kpc changes little at $|b| > 20°$: at 17 < G < 21 few halo-
  field stars have parallaxes measured that precisely. It matters more at low latitude.
- **Colour cut.** Keeping 0.3 < BP − RP < 1.3 recovers as many known objects with fewer spurious pixels; it favours old,
  metal-poor populations (Boötes I and Crater 2 become more significant) but suppresses objects with other colours — Gaia 1 and Gaia 2 lose most of their signal — and drops stars
  without BP/RP photometry. A filter matched to one population is blind to others: the same trade-off as the isochrone
  filters of Chapters 4–5.
- The spurious-pixel counts are a rough indicator only: they include real but uncatalogued structures, bright-star and
  galaxy artefacts, and the residual scanning pattern.

## Summary

- Database-side HEALPix histograms make all-sky density maps of 10⁹ stars cheap.
- The Koposov et al. (2017) statistic — kernel density minus its annulus mean, over the annulus scatter — reproduces their
  Figure 1 from Gaia DR1.
- Kernel size matters: compact clusters and extended dwarfs need different kernels.
- DR3 and its parallaxes and colours change both the recovery of known objects and the number of spurious peaks.